# Galileo, frozen encoder (variant-dependent embeddings)

This notebook follows `src/models/galileo.py` and shows how the wrapper converts the shared benchmark object into Galileo's grouped space-time inputs.

Key properties:
- Uses the NASA Harvest Galileo v1 wrapper vendored under `src/utils/galileoutil`.
- Supports `nano`, `tiny`, and `base` variants.
- Packs S2, S1, NDVI, climate, location, and time metadata into Galileo input groups.
- Pools the encoder output into one embedding per parcel.

## What Galileo expects as input

| Tensor | Shape | Meaning |
| --- | --- | --- |
| `x_st` | `(N, 1, 1, T, 13)` | space-time remote sensing channels |
| `sp_x` | `(N, 1, 1, 16)` | static spatial channels |
| `t_x` | `(N, T, 6)` | time-only metadata |
| `st_x` | `(N, 18)` | static metadata |
| masks | grouped shapes matching inputs | missing-input indicators expected by Galileo |
| output embedding | `(N, D)` | pooled frozen representation, where `D` depends on the variant |

The wrapper handles missing groups by filling values and marking them masked.

In [ ]:
import os
import sys
import importlib.util
from pathlib import Path
from types import SimpleNamespace

import numpy as np

REPO = Path.cwd()
while not (REPO / 'src').exists() and REPO.parent != REPO:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))


def load(name, rel):
    spec = importlib.util.spec_from_file_location(name, REPO / rel)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


In [ ]:
galileo = load('galileo_mod', 'src/models/galileo.py')

print('variants:', galileo.GALILEO_VARIANTS)
print('space-time bands:', galileo.SPACE_TIME_BANDS)


In [ ]:
rng = np.random.default_rng(7)
N, T = 4, 18

s2_bands = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12', 'NDVI']
s1_bands = ['VV', 'VH']
climate_bands = ['temperature', 'precipitation', 'elevation', 'slope']

bench = SimpleNamespace(
    name='synthetic',
    s2=(rng.random((N, T, len(s2_bands))) * 5000).astype('float32'),
    s1=rng.normal(-12, 3, size=(N, T, len(s1_bands))).astype('float32'),
    climate=rng.normal(0, 1, size=(N, T, len(climate_bands))).astype('float32'),
    s2_mask=np.ones((N, T), dtype='float32'),
    s1_mask=np.ones((N, T), dtype='float32'),
    climate_mask=np.ones((N, T), dtype='float32'),
    doy=np.tile(np.linspace(15, 350, T), (N, 1)).astype('float32'),
    latlon=np.repeat(np.array([[11.0, 79.0]], dtype='float32'), N, axis=0),
    years=np.full(N, 2021, dtype='int64'),
    s2_bands=s2_bands,
    s1_bands=s1_bands,
    climate_bands=climate_bands,
)

print('s2', bench.s2.shape, bench.s2_bands)
print('s1', bench.s1.shape, bench.s1_bands)
print('climate', bench.climate.shape, bench.climate_bands)
print('doy', bench.doy.shape, 'latlon', bench.latlon.shape)


## Wrapper mapping

`_bench_to_galileo` is the conversion used internally by `encode`. It returns all grouped Galileo inputs and masks without loading model weights.

In [ ]:
enc = galileo.GalileoEncoder(device='cpu', model_size='nano', patch_size=1)
inputs = enc._bench_to_galileo(bench)

names = ['x_st', 'sp_x', 't_x', 'st_x', 's_t_m', 'sp_m', 't_m', 'st_m', 'months']
for name, value in zip(names, inputs):
    print(name, value.shape, value.dtype)

print('embedding dim for nano:', enc.embedding_dim)


## Forward pass -> embedding

The real forward pass may download Galileo weights if they are not already cached. It is disabled by default so the notebook can be used as an input-shape reference first.

In [ ]:
RUN_REAL_FORWARD = False

enc = galileo.GalileoEncoder(device='cpu', model_size='nano', patch_size=1)

if not RUN_REAL_FORWARD:
    print('Set RUN_REAL_FORWARD = True to run the frozen encoder.')
else:
    emb = enc.encode(bench)
    print('embedding', emb.shape)
    print('first row, first 8 dims', np.round(emb[0, :8], 4))
